In [0]:
%run ./Classroom-Setup-Common-SQL-WAREHOUSE

In [0]:
-- Create (or replace) a reusable stored procedure that verifies a given
-- set of required tables exists in a given catalog.schema.
--
-- Parameters:
--   target_catalog  STRING         - the catalog to look in
--   target_schema   STRING         - the schema to look in
--   required_tables ARRAY<STRING>  - the list of tables that must exist
--
-- If any of the required tables are missing, raises an error listing the
-- missing tables and instructing the learner to run the required setup.
CREATE OR REPLACE PROCEDURE check_required_tables(
  IN target_catalog  STRING,
  IN target_schema   STRING,
  IN required_tables ARRAY<STRING>
)
  LANGUAGE SQL
  SQL SECURITY INVOKER
  COMMENT 'Verifies that the given array of tables exists in target_catalog.target_schema. Raises an error listing any that are missing.'
AS
BEGIN
  DECLARE missing ARRAY<STRING>;

  -- Set difference: required_tables minus tables that actually exist
  SET missing = array_except(
    required_tables,
    (SELECT collect_list(table_name)
     FROM IDENTIFIER(target_catalog || '.information_schema.tables')
     WHERE table_schema = target_schema)
  );

  IF size(missing) > 0 THEN
    SELECT raise_error(CONCAT(
      'Required tables missing in ',
      target_catalog, '.', target_schema, ': ',
      array_join(array_sort(missing), ', '),
      '. Please run the "Classroom-Setup-Required-Data" notebook in the Includes folder before continuing.'
    ));
  ELSE
    SELECT CONCAT(
      '✅ All required tables found in ',
      target_catalog, '.', target_schema,
      ' (', size(required_tables), ' tables).'
    ) AS `Status`;
  END IF;
END;

In [0]:
-- Run the check for this course. Pass the current catalog/schema and the
-- course's required table list. Reusable across courses — change the
-- array to match whatever tables each course needs.
CALL check_required_tables(
  target_catalog  => my_catalog,
  target_schema   => my_schema,
  required_tables => array('ca_customers', 'ca_opportunities', 'ca_orders', 'ca_products', 'ca_sales_targets')
);

In [0]:
SELECT current_catalog() AS `Course Catalog`,
       current_schema()  AS `Your Schema`;